In [ ]:
import pandas as pd
import numpy as np
from datetime import date

rng = np.random.default_rng(42)

N_CUSTOMERS = 300
START_DATE = pd.Timestamp('2024-10-01')
END_DATE   = pd.Timestamp('2026-09-30')

In [ ]:
# ---- word lists used to build plausible company names -------------------
PREFIXES = ['Brightline', 'Nordic', 'Anka', 'Ege', 'Crescent', 'Harbour',
            'Vantage', 'Kestrel', 'Meridian', 'Orchard', 'Ravenna', 'Silverbrook']
MIDDLES  = ['Supplies', 'Trading', 'Logistics', 'Foods', 'Systems',
            'Industries', 'Components', 'Solutions', 'Partners']
SUFFIXES = ['Ltd', 'PLC', 'AB', 'GmbH', 'AS', 'BV']

customer_id = np.arange(1, N_CUSTOMERS + 1)

# rng.choice picks randomly from a list; size = how many values we want
customer_name = [
    f"{p} {m} {s}"
    for p, m, s in zip(
        rng.choice(PREFIXES, size=N_CUSTOMERS),
        rng.choice(MIDDLES,  size=N_CUSTOMERS),
        rng.choice(SUFFIXES, size=N_CUSTOMERS),
    )
]

# p = probability of each option, so the book is UK-heavy like a real UK seller
country = rng.choice(
    ['UK', 'Germany', 'Sweden', 'Turkiye', 'Netherlands', 'France'],
    size=N_CUSTOMERS,
    p=[0.50, 0.12, 0.10, 0.12, 0.08, 0.08],
)

# 30 and 60 day terms are the most common in practice
payment_terms = rng.choice([30, 45, 60, 90], size=N_CUSTOMERS,
                           p=[0.45, 0.15, 0.30, 0.10])

# ---- payment behaviour: assigned per customer, used later for payments ---
# early  = pays before the due date
# ontime = pays around the due date
# late   = habitually 10-40 days late
# bad    = very late, often part-paid or not paid at all
payment_behaviour = rng.choice(['early', 'ontime', 'late', 'bad'],
                               size=N_CUSTOMERS,
                               p=[0.15, 0.45, 0.30, 0.10])

# ---- size tier: a few big customers make up most of the balance ----------
size_tier = rng.choice(['small', 'medium', 'large'], size=N_CUSTOMERS,
                       p=[0.60, 0.30, 0.10])

df_customers = pd.DataFrame({
    'customer_id': customer_id,
    'customer_name': customer_name,
    'country': country,
    'payment_terms': payment_terms,
    'payment_behaviour': payment_behaviour,
    'size_tier': size_tier,
})

df_customers.head(10)

In [ ]:
df_customers['country'].value_counts()
df_customers['payment_behaviour'].value_counts()
df_customers['size_tier'].value_counts()

In [ ]:
# How many invoices per customer per month, by size tier.
# Large customers invoice far more often and that's what creates the
# concentration you see in real ledgers.
INVOICES_PER_MONTH = {'small': 0.4, 'medium': 1.2, 'large': 3.5}

# Typical invoice value by tier (mean, spread) in GBP
INVOICE_VALUE = {'small': (1_200, 0.6), 'medium': (6_500, 0.5), 'large': (28_000, 0.45)}

# Months covered by the dataset
months = pd.date_range(START_DATE, END_DATE, freq='MS')

rows = []
invoice_id = 100_000

for _, cust in df_customers.iterrows():
    tier = cust['size_tier']
    base_rate = INVOICES_PER_MONTH[tier]
    mean_value, spread = INVOICE_VALUE[tier]

    for month_start in months:
        # Seasonality: busier autumn, quiet in summer and at Christmas
        month_factor = {1: 0.9, 2: 1.0, 3: 1.1, 4: 1.0, 5: 1.0, 6: 0.95,
                        7: 0.7, 8: 0.7, 9: 1.1, 10: 1.2, 11: 1.2, 12: 0.8}[month_start.month]

        # Poisson gives a whole number of invoices around the expected rate
        n_invoices = rng.poisson(base_rate * month_factor)

        for _ in range(n_invoices):
            invoice_id += 1

            # Random day within the month
            day_offset = rng.integers(0, month_start.days_in_month)
            invoice_date = month_start + pd.Timedelta(days=int(day_offset))
            if invoice_date > END_DATE:
                continue

            # Lognormal keeps values positive and right-skewed:
            # many ordinary invoices, a few very large ones.
            amount = float(rng.lognormal(mean=np.log(mean_value), sigma=spread))
            amount = round(amount, 2)

            due_date = invoice_date + pd.Timedelta(days=int(cust['payment_terms']))

            # Most invoicing is in GBP; overseas customers sometimes in local currency
            if cust['country'] in ('Germany', 'Netherlands', 'France'):
                currency = rng.choice(['GBP', 'EUR'], p=[0.4, 0.6])
            elif cust['country'] == 'Sweden':
                currency = rng.choice(['GBP', 'EUR'], p=[0.7, 0.3])
            else:
                currency = 'GBP'

            rows.append({
                'invoice_id': invoice_id,
                'customer_id': cust['customer_id'],
                'invoice_date': invoice_date.date(),
                'due_date': due_date.date(),
                'invoice_amount': amount,
                'currency': currency,
            })

df_invoices = pd.DataFrame(rows)
print(len(df_invoices), 'invoices')
df_invoices.head()

In [ ]:
df_invoices['invoice_amount'].describe()


In [ ]:
df_invoices.groupby(df_invoices['invoice_date'].astype('datetime64[ns]').dt.to_period('M')).size().head(12)

In [ ]:
# Payment delay in days, relative to the due date: (mean, spread)
# Negative mean = pays before it's due.
BEHAVIOUR_DELAY = {
    'early':  (-6,  4),
    'ontime': ( 4,  6),
    'late':   (22, 12),
    'bad':    (55, 28),
}

# Chance the invoice is simply never paid (write-off or long-running dispute)
NEVER_PAY_PROB = {'early': 0.00, 'ontime': 0.005, 'late': 0.02, 'bad': 0.10}

PART_PAY_PROB = 0.12   # chance the invoice is settled in two instalments

# quick lookup so we don't merge inside the loop
behaviour_by_customer = df_customers.set_index('customer_id')['payment_behaviour'].to_dict()

payment_rows = []
payment_id = 900_000

for inv in df_invoices.itertuples(index=False):
    behaviour = behaviour_by_customer[inv.customer_id]

    # some invoices are never paid at all
    if rng.random() < NEVER_PAY_PROB[behaviour]:
        continue

    mu, sigma = BEHAVIOUR_DELAY[behaviour]
    delay_days = int(round(rng.normal(mu, sigma)))

    due = pd.Timestamp(inv.due_date)
    invoice_dt = pd.Timestamp(inv.invoice_date)

    payment_date = due + pd.Timedelta(days=delay_days)
    # can't pay before the invoice exists
    payment_date = max(payment_date, invoice_dt + pd.Timedelta(days=1))

    # if the payment would land after our data window, the invoice is still open
    if payment_date > END_DATE:
        continue

    if rng.random() < PART_PAY_PROB:
        # instalment 1: 30-60% of the invoice
        share = float(rng.uniform(0.3, 0.6))
        first = round(inv.invoice_amount * share, 2)
        payment_id += 1
        payment_rows.append({
            'payment_id': payment_id, 'invoice_id': inv.invoice_id,
            'payment_date': payment_date.date(), 'payment_amount': first,
            'currency': inv.currency,
        })

        # instalment 2: the rest, some weeks later - unless that's beyond our window,
        # which leaves a part-paid invoice sitting open
        second_date = payment_date + pd.Timedelta(days=int(rng.integers(10, 50)))
        if second_date <= END_DATE:
            payment_id += 1
            payment_rows.append({
                'payment_id': payment_id, 'invoice_id': inv.invoice_id,
                'payment_date': second_date.date(),
                'payment_amount': round(inv.invoice_amount - first, 2),
                'currency': inv.currency,
            })
    else:
        payment_id += 1
        payment_rows.append({
            'payment_id': payment_id, 'invoice_id': inv.invoice_id,
            'payment_date': payment_date.date(),
            'payment_amount': inv.invoice_amount,
            'currency': inv.currency,
        })

df_payments = pd.DataFrame(payment_rows)
print(len(df_payments), 'payments')
df_payments.head()

In [ ]:
paid_ids = set(df_payments['invoice_id'])
open_invoices = df_invoices[~df_invoices['invoice_id'].isin(paid_ids)]
print(f"{len(open_invoices)} invoices with no payment ({len(open_invoices)/len(df_invoices):.1%})")

In [ ]:
from pathlib import Path

OUT = Path('data')
OUT.mkdir(exist_ok=True)

df_customers.drop(columns=['payment_behaviour', 'size_tier']).to_csv(
    OUT / 'customers.csv', index=False)
df_invoices.to_csv(OUT / 'invoices.csv', index=False)
df_payments.to_csv(OUT / 'payments.csv', index=False)

print('written to', OUT.resolve())